## EXTRACCIÓN DE DATOS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pprint as pp
import requests
import json
import time
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from rapidfuzz import process, fuzz

### Ficheros y Credenciales APIs

NOTA: Para conectarse a las diferentes APIs (Spotipy, LastFM, MusicBrainz, TMDB, OMDB) es necesario disponer de credenciales personales como desarrollador. Estas claves son gratuitas pero son personales e intransferibles y por esta razón no están compartidas en este repositorio público del proyecto.

In [20]:
# Movie Songs data files
MOVIE_SONGS_SAMPLE = 'data/movie_songs_sample.csv'
MOVIE_SONGS_JBONDS = 'data/movie_songs_jbonds.csv'
MOVIE_SONGS_COLLECTION = 'data/movie_songs_collected_500.csv'

# Spotify Authentication
SPOTIFY_CLIENT_ID = "XXX"
SPOTIFY_CLIENT_SECRET = "89f9b80e7bd74dc4945c14ae9d440b8d"
SPOTIFY_REDIRECT_URI = "http://127.0.0.1:8080"

auth_manager = SpotifyClientCredentials(client_id=SPOTIFY_CLIENT_ID,
                                        client_secret=SPOTIFY_CLIENT_SECRET)

sp = spotipy.Spotify(auth_manager=auth_manager)

# LastFM Credentials
LASTFM_API_KEY = "XXX"
LASTFM_BASE_URL = "http://ws.audioscrobbler.com/2.0/"

# MusicBrainz Setup
MB_HEADERS = { "User-Agent": "MoviSeongsDataProject/1.0 ( xxx@gmail.com )" }
MB_BASE_URL = "https://musicbrainz.org/ws/2"

# The Movie Database Credentials
TMDB_API_KEY = "XXX"
TMDB_BASE_URL = "https://api.themoviedb.org/3"

# The Open Movie Database Credentials
OMDB_API_KEY = "XXX"
OMDB_BASE_URL = "http://www.omdbapi.com/"

# Reference data files
ACCLAIMED_MUSIC_SONGS = 'data/acclaimed_music_top_songs.csv'
CHART_MASTERS_SONGS = 'data/chart_masters_top_songs.csv'
CHART_MASTERS_ARTISTS = 'data/chart_masters_top_artists.csv'

### Funciones Extracción Canciones

In [3]:
# Spotify Song Extraction
def fetch_best_spotify_track(song_title, artists, target_year):
    """
    Searches Spotify and prioritizes the track matching the target year
    """

    # search_artist = original_artists[0] if isinstance(original_artists, list) else original_artists
    # query = f"track:{song_title} artist:{search_artist}"

    # Ensure original_artists is a list
    if isinstance(artists, str):
        artists_to_try = [a.strip() for a in artists.split(',')]
    elif isinstance(artists, list):
        artists_to_try = artists
    else:
        artists_to_try = []

    tracks = []

    # Smart Query: Loop through available artists
    for search_artist in artists_to_try:
        print(f"  -> Searching Spotify for: '{song_title}' by {search_artist}...")
        query = f"track:{song_title} artist:{search_artist}"
        
        try:
            results = sp.search(q=query, type='track', limit=10)
            tracks = results['tracks']['items']
            
            if tracks:
                print(f"  -> Spotify: Match found using artist: '{search_artist}'")
                break # Success! Stop trying other artists
            else:
                print(f"  -> Spotify: No match for '{search_artist}'. Trying next...")
                time.sleep(1.0) # Respect rate limits before the next loop
                
        except spotipy.SpotifyException as e:
            print(f"  -> Spotify API Error on '{search_artist}': {e}")
            time.sleep(1.0)
            continue
            
    if not tracks:
        print("  -> Spotify: [FAILED] No match found for any listed artist.")
        return None
        
    # Helper for extraction data
    def extract_spotify_data(track_obj):
        return {
            "spotify_id": track_obj.get('id', 'N/A'),
            "spotify_track_title": track_obj.get('name', 'N/A'),
            "spotify_artists": [a.get('name') for a in track_obj.get('artists', [])], 
            "spotify_album_title": track_obj.get('album', {}).get('name', 'N/A'),
            "spotify_release_date": track_obj.get('album', {}).get('release_date', ''),
            "spotify_duration_ms": track_obj.get('duration_ms', 0),
            "isrc": track_obj.get('external_ids', {}).get('isrc', 'N/A')
        }

    # Check the top result's year
    top_track = tracks[0]
    top_year = top_track.get('album', {}).get('release_date', '')[:4]
    
    chosen_track = top_track
    
    if top_year == str(target_year):
        print(f"  -> Spotify: Top result matches the target year ({target_year}).")
    
    # Search fallback
    if top_year != str(target_year):
        target_year_int = int(target_year)
        exact_match_found = False
        
        # Priority 1: Look for an exact year match
        for track in tracks[1:]:
            track_year = track.get('album', {}).get('release_date', '')[:4]
            if track_year.isdigit() and int(track_year) == target_year_int:
                chosen_track = track
                print("  -> Spotify: Exact historical year match found!")
                exact_match_found = True
                break
                
        # Priority 2: Look for a close match (+/- 1 year) if exact fails
        if not exact_match_found:
            for track in tracks[1:]:
                track_year = track.get('album', {}).get('release_date', '')[:4]
                if track_year.isdigit() and abs(int(track_year) - target_year_int) <= 1:
                    chosen_track = track
                    print("  -> Spotify: Close historical year match found (+/- 1 year).")
                    break
                    
    return extract_spotify_data(chosen_track)


# LastFM Song Extraction
def fetch_best_lastfm_data(song_title, artist_names):
    """
    Iterates through a list of artists and returns the Last.fm data with the highest playcount
    """

    best_lastfm_data = None
    max_playcount = -1

    for artist in artist_names:
        print(f"  -> Querying Last.fm for: '{song_title}' by {artist}...")

        params = {
            "method": "track.getInfo",
            "api_key": LASTFM_API_KEY,
            "artist": artist,
            "track": song_title,
            "format": "json",
            "autocorrect": 1
        }

        try:
            response = requests.get(LASTFM_BASE_URL, params=params)
            data_json = response.json()

            if "error" in data_json:
                print(f"     [!] Not found under '{artist}'.")
                time.sleep(1)
                continue

            track = data_json.get("track", {})
            track_artist = track.get("artist", {})

            # Extract tags safely
            tags_list = track.get("toptags", {}).get("tag", [])
            top_tags = [tag["name"] for tag in tags_list[:3]]

            current_playcount = int(track.get("playcount", 0) or 0)

            # Dictionary Keys
            lastfm_data = {
                "lastfm_track_title": track.get("name"),
                "lastfm_artist_name": track_artist.get("name"),
                "lastfm_track_playcount": current_playcount,
                "lastfm_track_listeners": int(track.get("listeners", 0) or 0),
                "lastfm_tags": top_tags,
                "lastfm_track_url": track.get("url"),
                "lastfm_duration_ms": int(track.get("duration", 0) or 0),
                "track_mbid": track.get("mbid"),
                "artist_mbid": track_artist.get("mbid"),
                "lastfm_album_title": track.get("album", {}).get("title"),
                "lastfm_album_artist": track.get("album", {}).get("artist"),
                "wiki_summary": track.get("wiki", {}).get("summary")
            }

            if current_playcount > max_playcount:
                max_playcount = current_playcount
                best_lastfm_data = lastfm_data
                print(f"     [+] Found {current_playcount:,} plays. (New Highest!)")
            else:
                print(f"     [-] Found {current_playcount:,} plays. (Lower than max)")

            # Sleep to respect rate limits
            time.sleep(1.0)

        except Exception as e:
            print(f"     [!] Request failed for {artist}: {e}")
            time.sleep(1)
            continue

    return best_lastfm_data


# LastFM Artist Extraction
def fetch_lastfm_artist_info(artist_name, artist_mbid=None):
    """
    Queries the Last.fm artist.getInfo endpoint.
    Prioritizes the MBID if available for perfect accuracy, 
    but safely falls back to text search if not.
    """
    print(f"Querying Last.fm Artist Data for: '{artist_name}'...")
    
    # Safe blank dictionary if the search fails
    null_artist_data = {
        "artist_playcount": 0,
        "artist_listeners": 0,
        "artist_bio": "N/A"
    }
    
    # Set up the base parameters
    params = {
        "method": "artist.getInfo",
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1
    }
    
    # Prioritize the MusicBrainz ID
    if artist_mbid and str(artist_mbid).lower() != "nan":
        params["mbid"] = artist_mbid
    
    # Primary search or fallback
    params["artist"] = artist_name
    
    try:
        response = requests.get(LASTFM_BASE_URL, params=params)
        data = response.json()
        
        if "error" in data:
            print(f"  -> [FAILED] Last.fm Error: {data.get('message')}")
            return null_artist_data
            
        artist_full = data.get("artist", {})
        
        artist_data = {
            "artist_playcount": int(artist_full.get("stats", {}).get("playcount", 0) or 0),
            "artist_listeners": int(artist_full.get("stats", {}).get("listeners", 0) or 0),
            "artist_bio": artist_full.get("bio", {}).get("summary", "N/A")
        }
        
        print("  -> Success!")
        return artist_data
        
    except Exception as e:
        print(f"  -> [ERROR] Request failed: {e}")
        return null_artist_data


# Spotify + LastFM Song Extraction
def extract_spotify_lastfm_data(song_title, artists, target_year):
    """
    Combines Spotify and LastFM logic into a single dictionary
    """

    print(f"\n=== PROCESSING: '{song_title}' ({target_year}) ===")

    # STEP 1: Get the chosen Spotify data
    spotify_data = fetch_best_spotify_track(song_title, artists, target_year)

    if not spotify_data:
        print("Pipeline aborted: Could not find track on Spotify.")
        return None

    # STEP 2: Build the whole List of Artists (Originals + Spotify Discoveries)
    all_artists = artists.copy()
    for sp_artist in spotify_data["spotify_artists"]:
        if sp_artist not in all_artists:
            all_artists.append(sp_artist)

    selected_artists = all_artists[:3]

    # STEP 3: Get the best LastFM data
    lastfm_data = fetch_best_lastfm_data(song_title, selected_artists)

    # STEP 4: Merge the dictionaries
    master_dict = spotify_data.copy()

    if lastfm_data:
        master_dict.update(lastfm_data)
    else:
        # If LastFM fails, fill with safe null values matching requested keys
        null_lastfm = {
            "lastfm_track_title": None, 
            "lastfm_artist_name": None, 
            "lastfm_track_playcount": 0,
            "lastfm_track_listeners": 0, 
            "lastfm_tags": [], 
            "lastfm_track_url": None,
            "lastfm_duration": 0, 
            "track_mbid": None, 
            "artist_mbid": None,
            "lastfm_album_title": None, 
            "lastfm_album_artist": None, 
            "wiki_summary": None
        }
        master_dict.update(null_lastfm)

    return master_dict


# MusicBrainz Song Extraction
def fetch_musicbrainz_song_and_covers(song_title, composers, original_artists):
    """
    Searches MusicBrainz directly for the 'Work' (composition).
    Extracts the ISWC code, songwriters, total recordings, unique covers, 
    and the chronological timeline of those covers (excluding the original artist).
    """
    print(f"Searching MusicBrainz for: '{song_title}' (Composer: {composers})...")

    null_mb_data = {
        "mb_work_id": None, 
        "mb_work_title": None, 
        "mb_iswcs": [],
        "mb_composers": [], 
        "mb_lyricists": [], 
        "mb_total_recordings": 0,
        "mb_original_recordings_count": 0, 
        "mb_cover_recordings_count": 0,
        "mb_covers_performers_count": 0, 
        "mb_covers_performers_top": [],
        "mb_covers_earliest_year": None, 
        "mb_covers_earliest_performer": None,
        "mb_covers_latest_year": None, 
        "mb_covers_latest_performer": None
    }

    # --- STEP 1: SEARCH THE WORK DATABASE DIRECTLY ---
    search_endpoint = f"{MB_BASE_URL}/work"
    works = []

    if isinstance(composers, str):
        # If it's a comma-separated string, split it into a list
        composers_to_try = [c.strip() for c in composers.split(',')]
    elif isinstance(composers, list):
        composers_to_try = composers
    else:
        composers_to_try = []
    
    if isinstance(original_artists, str):
        original_artists_list = [a.strip() for a in original_artists.split(',')]
    elif isinstance(original_artists, list):
        original_artists_list = original_artists
    else:
        original_artists_list = []
    
    # ref_artist = original_artist[0] if isinstance(original_artist, list) else original_artist
    # search_query = f'work:"{song_title}" AND artist:"{composer_name}"'

    try:
        # Smart Query: Loop through available composers
        for comp in composers_to_try:
            if comp and comp.lower() != "unknown":
                search_query = f'work:"{song_title}" AND artist:"{comp}"'
                search_params = {"query": search_query, "fmt": "json", "limit": 3}
                
                search_response = requests.get(search_endpoint, headers=MB_HEADERS, params=search_params)
                search_data = search_response.json()
                works = search_data.get("works", [])
                
                if works:
                    print(f"  -> Match found using composer: '{comp}'")
                    break # Success! Stop trying other composers
                    
                print(f"  -> No match for '{comp}'. Trying next...")
                time.sleep(1.0) # Respect rate limits before the next loop

        # THE FALLBACK: If strict searches failed (or no valid composers were provided)
        if not works:
            print(f"  -> Strict searches failed. Retrying with just title: '{song_title}'...")
            time.sleep(1.2) # Ensure we don't hit the API too fast
            search_params = {"query": f'work:"{song_title}"', "fmt": "json", "limit": 3}
            search_response = requests.get(search_endpoint, headers=MB_HEADERS, params=search_params)
            search_data = search_response.json()
            works = search_data.get("works", [])

        if not works:
            print("  -> [FAILED] No matching works found on MusicBrainz.")
            return null_mb_data # Return safe empty dictionary instead of None

        # Grab the top result's ID
        work_id = works[0].get("id")
        official_title = works[0].get("title")
        print(f"  -> Found Composition: '{official_title}' (Work ID: {work_id})")

        # Respect rate limits before next call
        time.sleep(1.2)

        # --- STEP 2: GET FULL DETAILS & ISWC ---
        work_endpoint = f"{MB_BASE_URL}/work/{work_id}"
        work_params = {"fmt": "json", "inc": "artist-rels"}

        work_response = requests.get(work_endpoint, headers=MB_HEADERS, params=work_params)
        work_details = work_response.json()

        # Extract ISWCs
        iswcs = work_details.get("iswcs", [])

        # Extract Songwriters
        composers = []
        lyricists = []

        for rel in work_details.get("relations", []):
            if rel.get("target-type") == "artist":
                rel_type = rel.get("type")
                person_name = rel.get("artist", {}).get("name")

                if rel_type == "composer":
                    composers.append(person_name)
                elif rel_type == "lyricist":
                    lyricists.append(person_name)
                elif rel_type == "writer":
                    composers.append(person_name)
                    lyricists.append(person_name)

        composers = list(set(composers))
        lyricists = list(set(lyricists))

        # --- STEP 3: BROWSE RECORDINGS FOR COVERS ---
        print(f"  -> Scanning MusicBrainz for covers of '{official_title}'...")
        browse_endpoint = f"{MB_BASE_URL}/recording"

        limit = 100
        offset = 0
        total_recordings = 0
        
        # Differentiate between Original Artist vs Covers
        original_recordings_count = 0
        cover_recordings_count = 0
        cover_performers_counts = {}
        
        # Timeline Trackers (Exclusively for Covers now)
        earliest_cover_year = None
        latest_cover_year = None
        earliest_cover_performer = None
        latest_cover_performer = None

        while True:
            params = {
                "work": work_id,
                "inc": "artist-credits",
                "limit": limit,
                "offset": offset,
                "fmt": "json"
            }

            response = requests.get(browse_endpoint, headers=MB_HEADERS, params=params)
            data = response.json()

            if offset == 0:
                total_recordings = data.get("recording-count", 0)
                print(f"     [!] Found {total_recordings} total recordings.")

            recordings = data.get("recordings", [])
            if not recordings:
                break

            for rec in recordings:
                # Gather all performers for this specific recording
                current_performers = []
                for credit in rec.get("artist-credit", []):
                    performer_name = credit.get("name")
                    if performer_name and 'unknown' not in performer_name.lower():
                        current_performers.append(performer_name)
                
                # Create a clean string of the artists
                performer_string = ", ".join(current_performers) if current_performers else "Unknown"

                # Check if this is the original artist's recording
                # is_original_artist = ref_artist.lower() in performer_string.lower()
                # Check if this is the original artist's recording against ANY of the provided original artists
                is_original_artist = any(
                    orig_art.lower() in performer_string.lower() 
                    for orig_art in original_artists_list
                )
                
                if is_original_artist:
                    original_recordings_count += 1
                else:
                    cover_recordings_count += 1
                    
                    # Covers for our top 5 sample
                    for p in current_performers:
                        cover_performers_counts[p] = cover_performers_counts.get(p, 0) + 1

                    # Check the date, and update our trackers (ONLY FOR COVERS)
                    rec_date = rec.get("first-release-date")
                    if rec_date and len(rec_date) >= 4:
                        try:
                            rec_year = int(rec_date[:4])
                            
                            if earliest_cover_year is None or rec_year < earliest_cover_year:
                                earliest_cover_year = rec_year
                                earliest_cover_performer = performer_string 
                                
                            if latest_cover_year is None or rec_year > latest_cover_year:
                                latest_cover_year = rec_year
                                latest_cover_performer = performer_string 
                                
                        except ValueError:
                            pass 

            offset += limit
            if offset >= total_recordings:
                break

            time.sleep(1.2)

        # Sort performers and extract top 5
        sorted_performers = sorted(cover_performers_counts.items(), key=lambda item: item[1], reverse=True)
        top_sample = [artist[0] for artist in sorted_performers[:3]]

        # --- STEP 4: COMBINE ALL DATA ---
        mb_song_data = {
            "mb_work_id": work_id,
            "mb_work_title": work_details.get("title"),
            "mb_iswcs": iswcs,
            "mb_composers": composers,
            "mb_lyricists": lyricists,
            "mb_total_recordings": total_recordings,
            "mb_original_recordings_count": original_recordings_count,
            "mb_cover_recordings_count": cover_recordings_count,
            "mb_covers_performers_count": len(cover_performers_counts),
            "mb_covers_performers_top": top_sample,
            "mb_covers_earliest_year": earliest_cover_year,
            "mb_covers_earliest_performer": earliest_cover_performer, 
            "mb_covers_latest_year": latest_cover_year,
            "mb_covers_latest_performer": latest_cover_performer 
        }

        return mb_song_data

    except Exception as e:
        print(f"MusicBrainz API Error: {e}")
        return None
 

# Extract Spotify + LastFM + MusicBrainz Song Data

def extract_complete_song_data(title, artists, songwriters, year):
  
  song_data = extract_spotify_lastfm_data(title, artists, year)
  mb_data = fetch_musicbrainz_song_and_covers(title, songwriters, artists)
  
  if not song_data:
        return None

  complete_song_data = song_data.copy()
  complete_song_data.update(mb_data)

  return complete_song_data

### Funciones Extracción Películas

In [4]:
# TMDB Movie Extraction

def fetch_tmdb_movie_data(movie_title, target_year):
    """
    Searches TMDB for a movie using its Title and Year.
    If found, it makes a second call to get detailed financial/genre data.
    """
    print(f"Searching TMDB for: '{movie_title}' ({target_year})...")

    # --- STEP 1: SEARCH FOR THE MOVIE ID ---
    search_endpoint = f"{TMDB_BASE_URL}/search/movie"
    search_params = {
        "api_key": TMDB_API_KEY,
        "query": movie_title
        # "primary_release_year": year
    }

    try:
        # RETRY LOGIC: Try up to 3 times
        for attempt in range(3):
            search_response = requests.get(search_endpoint, params=search_params)
            
            if search_response.status_code == 200:
                break # Success! Break out of the retry loop
                
            print(f"  -> [WARNING] TMDB Search returned {search_response.status_code}. Retrying ({attempt+1}/3)...")
            time.sleep(2.0) # Wait 2 seconds before trying again
            
        # If it failed all 3 times, abort
        if search_response.status_code != 200:
            print("  -> [FAILED] TMDB Search server is unreachable after retries.")
            return None

        search_data = search_response.json()
        results = search_data.get('results', [])

        # If the broad title search returns absolutely nothing, abort the search
        if not results:
            print("  -> [FAILED] Movie not found on TMDB.")
            return None

        best_match = None
        target_year_int = int(target_year)

        # Priority 1: Look for an EXACT year match
        for movie in results:
            release_date = movie.get('release_date', '')
            if release_date and len(release_date) >= 4:
                if int(release_date[:4]) == target_year_int:
                    best_match = movie
                    print("  -> Found exact year match!")
                    break

        # Priority 2: Look for a CLOSE match (+/- 1 year) if exact fails
        if not best_match:
            for movie in results:
                release_date = movie.get('release_date', '')
                if release_date and len(release_date) >= 4:
                    if abs(int(release_date[:4]) - target_year_int) <= 1:
                        best_match = movie
                        print(f"  -> Found close year match (+/- 1 year discrepancy).")
                        break

        # Priority 3: Fallback to the top relevance result if years don't align at all
        if not best_match:
            best_match = results[0]
            print("  -> [!] No year match found. Falling back to top TMDB relevance result.")

        # Grab the ID of our winning match
        movie_id = best_match['id']
        official_title = best_match['title']
        print(f"  -> Selected TMDB Match: '{official_title}' (ID: {movie_id})")

        # Respect rate limits between API calls
        time.sleep(1.2)

        # --- STEP 2: GET FULL DETAILS ---
        # The search endpoint doesn't return budget/revenue, so use ID endpoint
        details_endpoint = f"{TMDB_BASE_URL}/movie/{movie_id}"
        details_params = {
            "api_key": TMDB_API_KEY,
            "append_to_response": "credits"
        }

        # RETRY LOGIC: Try up to 3 times
        for attempt in range(3):
            details_response = requests.get(details_endpoint, params=details_params)
            
            if details_response.status_code == 200:
                break # Success! Break out of the retry loop
                
            print(f"  -> [WARNING] TMDB Details returned {details_response.status_code}. Retrying ({attempt+1}/3)...")
            time.sleep(2.0)
            
        # If it failed all 3 times, abort
        if details_response.status_code != 200:
            print("  -> [FAILED] TMDB Details server is unreachable after retries.")
            return None
        
        details_data = details_response.json()

        genres_list = [genre['name'] for genre in details_data.get('genres', [])]
        companies_list = [comp['name'] for comp in details_data.get('production_companies', [])]

        # Extract Credits
        credits = details_data.get('credits', {})
        cast = credits.get('cast', [])
        crew = credits.get('crew', [])
        actors_list = [person['name'] for person in cast[:4]]
        directors_list = [person['name'] for person in crew if person.get('job') == 'Director']
        composers_list = [person['name'] for person in crew if person.get('job') in ['Original Music Composer', 'Music']]

        tmdb_data = {
            "tmdb_id": movie_id,
            "imdb_id": details_data.get('imdb_id'),
            "tmdb_official_title": details_data.get('title'),
            "tmdb_original_title": details_data.get('original_title'),
            "tmdb_release_date": details_data.get('release_date'),
            "tmdb_runtime": details_data.get('runtime', 0),
            "tmdb_budget": details_data.get('budget', 0),
            "tmdb_revenue": details_data.get('revenue', 0),
            "tmdb_vote_average": details_data.get('vote_average', 0.0),
            "tmdb_vote_count": details_data.get('vote_count', 0),
            "tmdb_original_language": details_data.get('original_language'),
            "tmdb_origin_countries": details_data.get('origin_country', []),
            "tmdb_actors": actors_list,
            "tmdb_directors": directors_list,
            "tmdb_composers": composers_list,
            "tmdb_production_companies": companies_list,
            "tmdb_genres": genres_list,
            "tmdb_tagline": details_data.get('tagline')
        }

        return tmdb_data

    except Exception as e:
        print(f"TMDB API Error: {e}")
        return None


# OMDB Movie Extraction

def fetch_omdb_movie_data(imdb_id):
    """
    Queries the OMDb API using an exact IMDb ID.
    Extracts Awards text, IMDb ratings, and Rotten Tomatoes scores.
    """
    if not imdb_id:
        print("  -> [FAILED] No IMDb ID provided for OMDb search.")
        return None

    print(f"Searching OMDb for IMDb ID: '{imdb_id}'...")

    # We pass 'i' for ID search
    params = {
        "apikey": OMDB_API_KEY,
        "i": imdb_id,
        "r": "json" # Request JSON response
    }

    try:
        response = requests.get(OMDB_BASE_URL, params=params)
        data = response.json()

        if data.get("Response") == "False":
            print(f"  -> [FAILED] OMDb Error: {data.get('Error')}")
            return None

        rotten_tomatoes_score = "N/A"
        for rating in data.get("Ratings", []):
            if rating.get("Source") == "Rotten Tomatoes":
                rotten_tomatoes_score = rating.get("Value")
                break

        omdb_data = {
            "omdb_title": data.get("Title", "N/A"),
            "omdb_awards": data.get("Awards", "N/A"),
            "omdb_imdb_rating": data.get("imdbRating", "N/A"),
            "omdb_imdb_votes": data.get("imdbVotes", "N/A"),
            "omdb_metascore": data.get("Metascore", "N/A"),
            "omdb_rotten_tomatoes": rotten_tomatoes_score
        }

        return omdb_data

    except Exception as e:
        print(f"OMDb API Error: {e}")
        return None


# Extract TMDB + OMDB Movie Data

def fetch_complete_movie_data(movie_title, target_year):
    # 1. Fetch from TMDB
    tmdb_data = fetch_tmdb_movie_data(movie_title, target_year)

    # 2. SAFETY CHECK: If TMDB failed, abort and return None immediately
    if not tmdb_data:
        return None

    complete_movie_data = tmdb_data.copy()

    # 4. Fetch from OMDb and merge
    imdb_id = tmdb_data.get("imdb_id")
    omdb_data = fetch_omdb_movie_data(imdb_id)
    
    if omdb_data:
        complete_movie_data.update(omdb_data)

    return complete_movie_data

### Preparar Dataframes del Modelo

Creamos el modelo de Datos a partir de un fichero CSV con la lista estructurada de Canciones + Películas, ordenadas por años, y con datos obtenidos a partir de las fuentes de información consultadas (Wikipedia, Academy Awards Website, Grammys, Google etc.).

In [7]:
# Import Movie Songs Data File
data_file = MOVIE_SONGS_COLLECTION

moviesongs_df = pd.read_csv(data_file)

display(moviesongs_df.head())
print(moviesongs_df.info())

# Generate Film IDs
moviesongs_df['film_id'] = moviesongs_df.groupby(['film_title', 'year'])['song_id'].transform('first')

# Create Data Frames

# Junction Table
df_songs_films = moviesongs_df[['song_id', 'year', 'song_title', 'film_title', 'film_id']].copy()
display(df_songs_films.head())

# Songs Table
songs_columns = [
    'song_id', 'year', 'song_title', 'song_composers', 'song_lyricists',
    'original_artists', 'streams_original', 'other_artists', 'streams_others',
    'oscar_song_nominee', 'oscar_song_win', 'grammy_song_nominee', 'grammy_song_win',
    'grammy_record_nominee', 'grammy_record_win'
]
songs_df = moviesongs_df[songs_columns].copy()

# Movies Table
movies_columns = [
    'film_id', 'year', 'film_title', 'film_directors', 'film_composers',
    'oscar_nominations', 'oscar_wins'
]
movies_df = moviesongs_df[movies_columns].copy()

# Drop duplicates so each movie only appears once!
movies_df = movies_df.drop_duplicates(subset=['film_id']).reset_index(drop=True)

,song_id,year,song_title,film_title,song_composers,song_lyricists,original_artists,streams_original,streams_others,other_artists,film_directors,film_composers,oscar_nominations,oscar_wins,oscar_song_nominee,oscar_song_win,grammy_song_nominee,grammy_song_win,grammy_record_nominee,grammy_record_win
0,193401,1934,The Continental,The Gay Divorcee,Con Conrad,Herb Magidson,"Fred Astaire, Ginger Rogers",65000.0,NaN,NaN,Mark Sandrich,"Kenneth Webb, Samuel Hoffenstein",5.0,1.0,Y,Y,N,N,N,N
1,193402,1934,Love In Bloom,She Loves Me Not,Ralph Rainger,Leo Robin,Bing Crosby,15800.0,171500.0,The Platters,Elliott Nugent,NaN,1.0,0.0,Y,N,N,N,N,N
2,193501,1935,Lullaby Of Broadway,Gold Diggers of 1935,Harry Warren,Al Dubin,Wini Shaw,37200.0,7020000.0,Doris Day,Busby Berkeley,NaN,2.0,1.0,Y,Y,N,N,N,N
3,193502,1935,Cheek To Cheek,Top Hat,Irving Berlin,Irving Berlin,Fred Astaire,28200000.0,210100000.0,Ella Fitzgerald,William A. Seiter,NaN,4.0,0.0,Y,N,N,N,N,N
4,193503,1935,Lovely To Look At,Roberta,Jerome Kern,"Dorothy Fields, Jimmy McHugh","Fred Astaire, Ginger Rogers, Irene Dunne",218000.0,NaN,NaN,William A. Seiter,NaN,1.0,0.0,Y,N,N,N,N,N


<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   song_id                500 non-null    int64  
 1   year                   500 non-null    int64  
 2   song_title             500 non-null    str    
 3   film_title             500 non-null    str    
 4   song_composers         500 non-null    str    
 5   song_lyricists         499 non-null    str    
 6   original_artists       499 non-null    str    
 7   streams_original       469 non-null    float64
 8   streams_others         104 non-null    float64
 9   other_artists          115 non-null    str    
 10  film_directors         500 non-null    str    
 11  film_composers         307 non-null    str    
 12  oscar_nominations      497 non-null    float64
 13  oscar_wins             497 non-null    float64
 14  oscar_song_nominee     498 non-null    str    
 15  oscar_song_win   

,song_id,year,song_title,film_title,film_id
0,193401,1934,The Continental,The Gay Divorcee,193401
1,193402,1934,Love In Bloom,She Loves Me Not,193402
2,193501,1935,Lullaby Of Broadway,Gold Diggers of 1935,193501
3,193502,1935,Cheek To Cheek,Top Hat,193502
4,193503,1935,Lovely To Look At,Roberta,193503


### Extraccion Datos Canciones

In [8]:
# Extract Songs Data

print("=== STARTING SONGS DATA EXTRACTION ===")
api_results = []

# Loop through the SONGS dataframe
for index, row in songs_df.iterrows():
    song_id = row['song_id']
    song_title = row['song_title']
    year = row['year']

    # Parse Artists
    if pd.notna(row['original_artists']):
        # Split by comma and strip any whitespace to ensure clean artist names
        original_artists = [artist.strip() for artist in str(row['original_artists']).split(',')]
    else:
        original_artists = []

    if pd.notna(row['other_artists']):
        other_artists = [artist.strip() for artist in str(row['other_artists']).split(',')]
    else:
        other_artists = []

    all_artists = list(dict.fromkeys(original_artists + other_artists))

    # Parse Composers
    if pd.notna(row['song_composers']):
        composers = [composer.strip() for composer in str(row['song_composers']).split(',')]
    else:
        composers = []

    # Parse Lyricists
    if pd.notna(row['song_lyricists']):
        lyricists = [lyricist.strip() for lyricist in str(row['song_lyricists']).split(',')]
    else:
        lyricists = []
    
    # Combine Writers & Deduplicate (Preserving Order)
    all_composers = list(dict.fromkeys(composers + lyricists))

    # API function
    fetched_data = extract_complete_song_data(song_title, all_artists, all_composers, year)

    # Unique song_id
    if fetched_data:
        fetched_data['song_id'] = song_id
        api_results.append(fetched_data)

print("\n=== EXTRACTION COMPLETE. COMMENCING MERGE ===")

# Convert the api results list of dictionaries into a new temporary DataFrame
api_df = pd.DataFrame(api_results)

# Merge the original songs_df with the new api_df using 'song_id'
df_songs_enriched = pd.merge(songs_df, api_df, on='song_id', how='left')

# Inspect the Enriched data
print("\n=== ENRICHED SONGS TABLE ===")
print(df_songs_enriched[['song_id', 'song_title', 'spotify_id', 'mb_iswcs', 'lastfm_track_playcount']].head())
print(f"\nTotal Rows in Enriched Database: {len(df_songs_enriched)}")

=== STARTING SONGS DATA EXTRACTION ===

=== PROCESSING: 'The Continental' (1934) ===
  -> Searching Spotify for: 'The Continental' by Fred Astaire...
  -> Spotify: Match found using artist: 'Fred Astaire'
  -> Querying Last.fm for: 'The Continental' by Fred Astaire...
     [+] Found 19,967 plays. (New Highest!)
  -> Querying Last.fm for: 'The Continental' by Ginger Rogers...
     [-] Found 1,387 plays. (Lower than max)
Searching MusicBrainz for: 'The Continental' (Composer: ['Con Conrad', 'Herb Magidson'])...
  -> Match found using composer: 'Con Conrad'
  -> Found Composition: 'The Continental' (Work ID: cbba508e-fc50-351f-9f1b-bab5a07d33e1)
  -> Scanning MusicBrainz for covers of 'The Continental'...
     [!] Found 141 total recordings.

=== PROCESSING: 'Love In Bloom' (1934) ===
  -> Searching Spotify for: 'Love In Bloom' by Bing Crosby...
  -> Spotify: Match found using artist: 'Bing Crosby'
  -> Querying Last.fm for: 'Love In Bloom' by Bing Crosby...
     [+] Found 6,003 plays. (N

### Datos Acclaimed Music y Chart Masters

In [10]:
df_am_songs = pd.read_csv(ACCLAIMED_MUSIC_SONGS)
df_cm_songs = pd.read_csv(CHART_MASTERS_SONGS)

# Regex Normalization
def normalize_titles(series):
    """Vectorized cleaning to strip punctuation and metadata in parentheses/brackets."""
    return (
        series.str.lower()
        .str.replace(r'[\(\[].*?[\)\]]', '', regex=True) 
        .str.replace(r'[^\w\s]', '', regex=True)         
        .str.strip()
    )

df_songs_ranked = df_songs_enriched.copy()
df_songs_ranked['match_title'] = normalize_titles(df_songs_ranked['song_title'])
df_am_songs['match_title'] = normalize_titles(df_am_songs['Song'])
df_cm_songs['match_title'] = normalize_titles(df_cm_songs['Song'])


# Create Combined Match Keys (Primary Artist + Title)
df_songs_ranked['match_key'] = (
    df_songs_ranked['original_artists']
    .fillna('')
    .str.split(',')
    .str[0]
    .str.lower()
    .str.strip() 
    + " " + 
    df_songs_ranked['match_title']
)

df_am_songs['match_key'] = (
    df_am_songs['Artist']
    .fillna('')
    .str.split(',')
    .str[0]
    .str.lower()
    .str.strip() 
    + " " + 
    df_am_songs['match_title']
)

df_cm_songs['match_key'] = (
    df_cm_songs['Artist']
    .fillna('')
    .str.split(',')
    .str[0]
    .str.lower()
    .str.strip() 
    + " " + 
    df_cm_songs['match_title']
)

"""
# Note: Assuming the raw CSV column is 'Artist' before you rename it later
df_am_songs['match_key'] = df_am_songs['Artist'].str.lower().str.strip() + " " + df_am_songs['match_title']
df_cm_songs['match_key'] = df_cm_songs['Artist'].str.lower().str.strip() + " " + df_cm_songs['match_title']
"""

# Fuzzy Matching Helper
def get_best_fuzzy_match(key, choices, default_thresh=90):
    """
    Uses token_sort_ratio to compare the full string. Demands a 95% match 
    for short combined keys (15 characters or less) to prevent collisions.
    """
    if pd.isna(key): 
        return None
    
    # Increased the strict length boundary to 15 since we are now checking Artist + Title
    threshold = 95 if len(str(key)) <= 15 else default_thresh
        
    match = process.extractOne(key, choices, scorer=fuzz.token_sort_ratio)
    
    if match and match[1] >= threshold:
        return match[0] 
        
    return key

# Extract lists of unique combined keys to match against
am_choices = df_am_songs['match_key'].dropna().unique().tolist()
cm_choices = df_cm_songs['match_key'].dropna().unique().tolist()

# Apply fuzzy mapping to our main dataset using the combined key
df_songs_ranked['match_key_am'] = df_songs_ranked['match_key'].apply(
    lambda x: get_best_fuzzy_match(x, am_choices)
)
df_songs_ranked['match_key_cm'] = df_songs_ranked['match_key'].apply(
    lambda x: get_best_fuzzy_match(x, cm_choices)
)

# Prepare Clean Dataframes
df_am_clean = (
    df_am_songs.drop_duplicates(subset=['match_key'])
    .rename(columns={'Ranking': 'am_ranking', 'Artist': 'am_song_artist', 'Year': 'am_song_year'})
    [['match_key', 'am_ranking', 'am_song_artist', 'am_song_year']]
)

df_cm_clean = (
    df_cm_songs.drop_duplicates(subset=['match_key'])
    .rename(columns={'Ranking': 'cm_ranking', 'Artist': 'cm_song_artist', 'EAS': 'cm_song_eas'})
    [['match_key', 'cm_ranking', 'cm_song_artist', 'cm_song_eas']]
)

# Strip commas from the ChartMasters strings before merging
df_cm_clean['cm_ranking'] = df_cm_clean['cm_ranking'].astype(str).str.replace(',', '', regex=False)
df_cm_clean['cm_song_eas'] = df_cm_clean['cm_song_eas'].astype(str).str.replace(',', '', regex=False)


# Merge (Using the new fuzzy match keys)
df_songs_ranked = pd.merge(
    df_songs_ranked, df_am_clean, 
    left_on='match_key_am', right_on='match_key', how='left'
).drop(columns=['match_key_y']).rename(columns={'match_key_x': 'match_key'})

df_songs_ranked = pd.merge(
    df_songs_ranked, df_cm_clean, 
    left_on='match_key_cm', right_on='match_key', how='left'
).drop(columns=['match_key_y', 'match_key_x']) 


# Cleanup and Type Casting
df_songs_ranked['am_ranking'] = pd.to_numeric(df_songs_ranked['am_ranking'], errors='coerce').astype('Int64')
df_songs_ranked['am_song_year'] = pd.to_numeric(df_songs_ranked['am_song_year'], errors='coerce').astype('Int64')
df_songs_ranked['cm_ranking'] = pd.to_numeric(df_songs_ranked['cm_ranking'], errors='coerce').astype('Int64')
df_songs_ranked['cm_song_eas'] = pd.to_numeric(df_songs_ranked['cm_song_eas'], errors='coerce').astype('Int64')


# Drop all the temporary fuzzy matching columns
cols_to_drop = ['match_title'] # 'match_key_am', 'match_key_cm'
df_songs_ranked = df_songs_ranked.drop(columns=cols_to_drop, errors='ignore')

In [11]:
display(df_songs_ranked[['song_title', 'match_key_am', 'match_key_cm']])
print("Acclaimed Music missing:", df_songs_ranked['am_ranking'].isna().sum())
print("ChartMasters missing:", df_songs_ranked['cm_ranking'].isna().sum())

,song_title,match_key_am,match_key_cm
0,The Continental,fred astaire the continental,fred astaire the continental
1,Love In Bloom,bing crosby love in bloom,bing crosby love in bloom
2,Lullaby Of Broadway,wini shaw lullaby of broadway,wini shaw lullaby of broadway
3,Cheek To Cheek,fred astaire cheek to cheek,fred astaire cheek to cheek
4,Lovely To Look At,fred astaire lovely to look at,fred astaire lovely to look at
...,...,...,...
495,Golden,ejae golden,ejae golden
496,I Lied To You,miles caton i lied to you,miles caton i lied to you
497,Train Dreams,sierra ferrell train dreams,sierra ferrell train dreams
498,Sweet Dreams Of Joy,nicholas pike sweet dreams of joy,nicholas pike sweet dreams of joy


Acclaimed Music missing: 416
ChartMasters missing: 425


### Extracción Datos Películas

In [12]:
# Extract Movies Data
print("\n=== STARTING MOVIES DATA EXTRACTION ===")
movie_api_results = []

for index, row in movies_df.iterrows():
    film_id = row['film_id']
    film_title = row['film_title']
    year = row['year']

    fetched_movie_data = fetch_complete_movie_data(film_title, year)

    # Unique film_id
    if fetched_movie_data:
        fetched_movie_data['film_id'] = film_id
        movie_api_results.append(fetched_movie_data)

print("\n=== MOVIE EXTRACTION COMPLETE. COMMENCING MERGE ===")
movie_api_df = pd.DataFrame(movie_api_results)
df_movies_enriched = pd.merge(movies_df, movie_api_df, on='film_id', how='left')

# Inspect the extracted Movie Data
print("\n=== ENRICHED MOVIES TABLE ===")
print(df_movies_enriched[['film_id', 'film_title', 'tmdb_id', 'omdb_imdb_rating', 'omdb_awards']].head())
print(f"\nTotal Rows in Enriched Movies Database: {len(df_movies_enriched)}")


=== STARTING MOVIES DATA EXTRACTION ===
Searching TMDB for: 'The Gay Divorcee' (1934)...
  -> Found exact year match!
  -> Selected TMDB Match: 'The Gay Divorcee' (ID: 28288)
Searching OMDb for IMDb ID: 'tt0025164'...
Searching TMDB for: 'She Loves Me Not' (1934)...
  -> Found exact year match!
  -> Selected TMDB Match: 'She Loves Me Not' (ID: 111582)
Searching OMDb for IMDb ID: 'tt0025774'...
Searching TMDB for: 'Gold Diggers of 1935' (1935)...
  -> Found exact year match!
  -> Selected TMDB Match: 'Gold Diggers of 1935' (ID: 43892)
Searching OMDb for IMDb ID: 'tt0026421'...
Searching TMDB for: 'Top Hat' (1935)...
  -> Found exact year match!
  -> Selected TMDB Match: 'Top Hat' (ID: 3080)
Searching OMDb for IMDb ID: 'tt0027125'...
Searching TMDB for: 'Roberta' (1935)...
  -> Found exact year match!
  -> Selected TMDB Match: 'Roberta' (ID: 31798)
Searching OMDb for IMDb ID: 'tt0026942'...
Searching TMDB for: 'Swing Time' (1936)...
  -> Found exact year match!
  -> Selected TMDB Match:

In [13]:
print(df_movies_enriched.head().to_string())

   film_id  year            film_title     film_directors                    film_composers  oscar_nominations  oscar_wins  tmdb_id    imdb_id   tmdb_official_title   tmdb_original_title tmdb_release_date  tmdb_runtime  tmdb_budget  tmdb_revenue  tmdb_vote_average  tmdb_vote_count tmdb_original_language tmdb_origin_countries                                                        tmdb_actors       tmdb_directors                                                      tmdb_composers                      tmdb_production_companies               tmdb_genres                                                   tmdb_tagline            omdb_title                                           omdb_awards omdb_imdb_rating omdb_imdb_votes omdb_metascore omdb_rotten_tomatoes
0   193401  1934      The Gay Divorcee      Mark Sandrich  Kenneth Webb, Samuel Hoffenstein                5.0         1.0    28288  tt0025164      The Gay Divorcee      The Gay Divorcee        1934-10-12           105       520000     

### Extracción de Datos Artistas

In [14]:
# LastFM Artist Extraction

def fetch_lastfm_artist_info(artist_name, artist_mbid=None):
    """
    Queries the Last.fm artist.getInfo endpoint.
    Prioritizes the MBID if available for perfect accuracy, 
    but safely falls back to text search if not.
    """
    print(f"Querying Last.fm Artist Data for: '{artist_name}'...")
    
    null_artist_data = {
        "artist_playcount": 0,
        "artist_listeners": 0,
        "artist_bio": "N/A"
    }
    
    params = {
        "method": "artist.getInfo",
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1
    }
    
    # Prioritize the MusicBrainz ID if we have it!
    if artist_mbid and str(artist_mbid).lower() != "nan":
        params["mbid"] = artist_mbid
    
    # Include text name as a primary search or fallback
    params["artist"] = artist_name
    
    try:
        response = requests.get(LASTFM_BASE_URL, params=params)
        data = response.json()
        
        if "error" in data:
            print(f"  -> [FAILED] Last.fm Error: {data.get('message')}")
            return null_artist_data
            
        artist_full = data.get("artist", {})
        
        artist_data = {
            "artist_playcount": int(artist_full.get("stats", {}).get("playcount", 0) or 0),
            "artist_listeners": int(artist_full.get("stats", {}).get("listeners", 0) or 0),
            "artist_bio": artist_full.get("bio", {}).get("summary", "N/A")
        }
        
        print("  -> Success!")
        return artist_data
        
    except Exception as e:
        print(f"  -> [ERROR] Request failed: {e}")
        return null_artist_data


# MusicBrainz Artist Extraction

def fetch_musicbrainz_artist_info(artist_name, artist_mbid=None):
    """
    Queries MusicBrainz for detailed Artist metadata.
    Prioritizes an exact MBID if provided, otherwise performs a text search.
    """
    
    null_artist_data = {
        "mb_artist_id": None, "mb_artist_type": None, "mb_gender": None,
        "mb_country": None, "mb_begin_area": None, "mb_disambiguation": None,
        "mb_lifespan_begin": None, "mb_lifespan_end": None, "mb_is_active": None,
        "mb_isnis": [], "mb_aliases": [], "mb_tags": [],
        "mb_wikidata_url": None, "mb_imdb_url": None
    }
    
    target_mbid = artist_mbid
    
    # --- STEP 1: RESOLVE THE MBID ---
    if not target_mbid or str(target_mbid).lower() == "nan":
        search_endpoint = f"{MB_BASE_URL}/artist"
        search_params = {"query": f'artist:"{artist_name}"', "fmt": "json", "limit": 3}
        
        try:
            search_response = requests.get(search_endpoint, headers=MB_HEADERS, params=search_params)
            search_data = search_response.json()
            
            artists = search_data.get("artists", [])
            if not artists:
                return null_artist_data
                
            target_mbid = artists[0].get("id")
            time.sleep(1.2) 
            
        except Exception as e:
            print(f"  -> [ERROR] Search API failed: {e}")
            return null_artist_data

    # --- STEP 2: GET FULL DETAILS ---
    details_endpoint = f"{MB_BASE_URL}/artist/{target_mbid}"
    details_params = {"fmt": "json", "inc": "aliases+tags+url-rels"}
    
    try:
        response = requests.get(details_endpoint, headers=MB_HEADERS, params=details_params)
        artist_details = response.json()
        
        # Extract Basic Demographics safely using "or {}"
        lifespan = artist_details.get("life-span") or {}
        area = artist_details.get("area") or {}
        begin_area = artist_details.get("begin-area") or {}
        
        # Extract Aliases safely using "or []"
        aliases = []
        for alias in (artist_details.get("aliases") or []):
            if alias.get("name"):
                aliases.append(alias.get("name"))
                
        # Extract Tags (Genres/Descriptors)
        tags = []
        for tag in (artist_details.get("tags") or []):
            if tag.get("name"):
                tags.append(tag.get("name"))
                
        # Extract External URLs
        wikidata_url = None
        imdb_url = None
        
        for rel in (artist_details.get("relations") or []):
            if rel.get("target-type") == "url":
                rel_type = rel.get("type")
                url = (rel.get("url") or {}).get("resource")
                
                if rel_type == "wikidata": wikidata_url = url
                elif rel_type == "imdb": imdb_url = url

        # Final dictionary
        extracted_data = {
            "mb_artist_id": target_mbid, 
            "mb_artist_type": artist_details.get("type"), 
            "mb_gender": artist_details.get("gender"), 
            "mb_country": area.get("name"), 
            "mb_begin_area": begin_area.get("name"), 
            "mb_disambiguation": artist_details.get("disambiguation"), 
            "mb_lifespan_begin": lifespan.get("begin"),
            "mb_lifespan_end": lifespan.get("end"),
            "mb_is_active": not lifespan.get("ended", False), 
            "mb_isnis": artist_details.get("isnis") or [], 
            "mb_aliases": list(set(aliases))[:5], 
            "mb_tags": tags[:5], 
            "mb_wikidata_url": wikidata_url,
            "mb_imdb_url": imdb_url
        }
        
        return extracted_data
        
    except Exception as e:
        print(f"  -> [ERROR] Details API failed: {e}")
        return null_artist_data


# Extract LastFM + MusicBrainz Artist Data

def extract_complete_artist_info(artist_name, artist_mbid=None):
    lastfm_data = fetch_lastfm_artist_info(artist_name, artist_mbid)
    mb_data = fetch_musicbrainz_artist_info(artist_name, artist_mbid)
    
    if not lastfm_data:
        return None

    artist_data = lastfm_data.copy()
    artist_data.update(mb_data)
    
    return artist_data

In [15]:
# ARTISTS Table

# Artist IDs
def generate_artist_id(artist_name):
    """
    Replicates the Pandas ID generation logic using pure Python string methods.
    1. Lowercase 2. Keep only alphanumeric/spaces 3. Collapse spaces to underscores
    """
    clean_name = str(artist_name).lower()
    
    # Keep only characters that are alphanumeric (letters/numbers) or spaces
    clean_name = "".join(char for char in clean_name if char.isalnum() or char.isspace())
    
    # .split() automatically strips outer spaces AND groups multiple inner spaces together.
    # We then join those grouped words with a single underscore.
    artist_id = "_".join(clean_name.split())
    
    return artist_id

# ==========================================
# CREATE THE COMPLETE ARTISTS TABLE
# ==========================================
print("\n=== STARTING ARTISTS DATA EXTRACTION ===")

artist_roles_dict = {}

# Mapping (DataFrame, Column Name, Role)
# (Assumes df_songs_ranked and df_movies_enriched are already loaded in the notebook)
role_mappings = [
    # Performers
    (df_songs_ranked, 'original_artists', 'performer'),
    (df_songs_ranked, 'other_artists', 'performer'),
    (df_songs_ranked, 'spotify_artists', 'performer'),
    (df_songs_ranked, 'lastfm_artist_name', 'performer'),
    # (df_songs_ranked, 'mb_covers_performers_top', 'performer')
    # (df_songs_ranked, 'mb_covers_earliest_performer', 'performer'),
    # (df_songs_ranked, 'mb_covers_latest_performer', 'performer')
    
    # Songwriters
    # (df_songs_ranked, 'song_composers', 'song_composer'),
    # (df_songs_ranked, 'song_lyricists', 'song_lyricist'),
    # Score Composers and Directors (From the Movies DataFrame),
    # (df_movies_enriched, 'film_composers', 'score_composer'),
    # (df_movies_enriched, 'film_directors', 'film_director') 
]

# Loop through mapping blueprint
for df, col, assigned_role in role_mappings:
    if col in df.columns:
        # Loop through every non-null value in the column
        for val in df[col].dropna():
            # Standardize everything into a list
            if isinstance(val, list):
                items = val
            elif isinstance(val, str):
                items = val.split(',')
            else:
                items = []
                
            for item in items:
                clean_name = str(item).strip()
                if clean_name:
                    # If this is a new artist, create an empty set
                    if clean_name not in artist_roles_dict:
                        artist_roles_dict[clean_name] = set()
                    
                    # Add the role to their set
                    artist_roles_dict[clean_name].add(assigned_role)

# Convert dictionary into list of rows
artist_rows = []
for name, roles_set in artist_roles_dict.items():
    artist_rows.append({
        'artist_name': name,
        # Convert the set to a sorted list, then join into comma-separated string
        'roles': ', '.join(sorted(list(roles_set)))
    })

# Create the DataFrame and sort it alphabetically
df_artists = pd.DataFrame(artist_rows).sort_values('artist_name').reset_index(drop=True)

# Artist ID's
df_artists['artist_id'] = df_artists['artist_name'].apply(generate_artist_id)

# ==========================================
# MAP Music Brainz IDs and init API columns
# ==========================================
mbid_mapping = {}
if 'lastfm_artist_name' in df_songs_ranked.columns and 'artist_mbid' in df_songs_ranked.columns:
    valid_mbids = df_songs_ranked.dropna(subset=['lastfm_artist_name', 'artist_mbid'])
    for _, row in valid_mbids.iterrows():
        clean_name = str(row['lastfm_artist_name']).strip()
        if clean_name:
            mbid_mapping[clean_name] = row['artist_mbid']

df_artists['mb_artist_id'] = df_artists['artist_name'].map(mbid_mapping)

# Reorder columns
df_artists = df_artists[['artist_id', 'artist_name', 'roles', 'mb_artist_id']]

# Initialize future API columns
print("Initializing schema columns for future API enrichment...")

future_api_columns = [
    # Last.fm Columns
    "artist_playcount", "artist_listeners", "artist_bio",
    # MusicBrainz Columns
    "mb_artist_type", "mb_gender", "mb_country", "mb_begin_area",
    "mb_disambiguation", "mb_lifespan_begin", "mb_lifespan_end",
    "mb_is_active", "mb_isnis", "mb_aliases", "mb_tags",
    "mb_wikidata_url", "mb_imdb_url"
]

# Safely represent missing data until the APIs populate it
for col in future_api_columns:
    df_artists[col] = pd.NA

# Update or Insert Artist Data

def update_or_add_artist(artists_df, artist_name, artist_mbid=None):
    """
    Takes an artist name, fetches their data, and dynamically updates 
    the existing Artists DataFrame or appends a new row if they don't exist.
    """
    print(f"\nProcessing Artist Request: '{artist_name}'")
    
    # 1. Generate the standardized ID
    artist_id = generate_artist_id(artist_name)
    
    # 2. Fetch the data
    fetched_data = extract_complete_artist_info(artist_name, artist_mbid)
    
    if not fetched_data:
        print(f"  -> [WARNING] No data returned for {artist_name}. Skipping update.")
        return artists_df
        
    # 3. Check if the Artist exists in DataFrame
    if artist_id in artists_df['artist_id'].values:
        print(f"  -> [UPDATE] Artist '{artist_id}' found in table. Updating columns...")
        
        row_idx = artists_df[artists_df['artist_id'] == artist_id].index[0]
        
        if artist_mbid and pd.isna(artists_df.at[row_idx, 'mb_artist_id']):
            artists_df.at[row_idx, 'mb_artist_id'] = artist_mbid
            
        # Loop through the fetched data and update the specific columns
        for key, value in fetched_data.items():
            if key in artists_df.columns:
                artists_df.at[row_idx, key] = value
                
    else:
        print(f"  -> [INSERT] Artist '{artist_id}' not found. Creating new row...")
        
        # Base dictionary
        new_row_data = {
            'artist_id': artist_id,
            'artist_name': artist_name,
            'roles': '',
            'mb_artist_id': artist_mbid if artist_mbid else pd.NA
        }
        
        # Merge the fetched API data
        new_row_data.update(fetched_data)
        
        # Convert the dictionary to a single-row DataFrame
        new_row_df = pd.DataFrame([new_row_data])
        
        # Ensure the new row has all the exact columns as the master table (fills missing with NaN)
        new_row_df = new_row_df.reindex(columns=df_artists.columns)
        
        # Safely concatenate the new row to the bottom of the master table
        artists_df = pd.concat([artists_df, new_row_df], ignore_index=True)
        
    return artists_df


# Extract all Artists Data

def extract_artists_data(artists_df):
    """
    Loops through every row in the Artists DataFrame, safely calling the 
    upsert function to fetch API data and update the table.
    """
    total_artists = len(artists_df)
    print(f"\n=== STARTING BATCH EXTRACTION FOR {total_artists} ARTISTS ===")
    
    # Extract the names and MBIDs to a list of dictionaries first
    artists_to_process = artists_df[['artist_name', 'mb_artist_id', 'artist_playcount']].to_dict('records')
    
    for i, row in enumerate(artists_to_process, 1):
        artist_name = row['artist_name']
        
        if pd.notna(row.get('artist_playcount')):
            print(f"  -> Skipping '{artist_name}': Data already exists in the table.")
            continue

        artist_mbid = None if pd.isna(row['mb_artist_id']) else row['mb_artist_id']
        
        print(f"\n--- Batch Process ({i}/{total_artists}) ---")
        
        # Call upsert function
        artists_df = update_or_add_artist(artists_df, artist_name, artist_mbid)
        
        time.sleep(1.0)
        
    print("\n=== BATCH EXTRACTION COMPLETE ===")
    return artists_df

# Populate Artists Table
df_artists = extract_artists_data(df_artists)

print("\n=== EXTRACTED ARTISTS ===")
print(df_artists.head(20)[['artist_id', 'roles', 'mb_country', 'artist_playcount']])


=== STARTING ARTISTS DATA EXTRACTION ===
Initializing schema columns for future API enrichment...

=== STARTING BATCH EXTRACTION FOR 612 ARTISTS ===

--- Batch Process (1/612) ---

Processing Artist Request: '*NSYNC'
Querying Last.fm Artist Data for: '*NSYNC'...
  -> Success!
  -> [UPDATE] Artist 'nsync' found in table. Updating columns...

--- Batch Process (2/612) ---

Processing Artist Request: '-M-'
Querying Last.fm Artist Data for: '-M-'...
  -> Success!
  -> [UPDATE] Artist 'm' found in table. Updating columns...

--- Batch Process (3/612) ---

Processing Artist Request: 'A-ha'
Querying Last.fm Artist Data for: 'A-ha'...
  -> Success!
  -> [UPDATE] Artist 'aha' found in table. Updating columns...

--- Batch Process (4/612) ---

Processing Artist Request: 'A. R. Rahman'
Querying Last.fm Artist Data for: 'A. R. Rahman'...
  -> Success!
  -> [UPDATE] Artist 'a_r_rahman' found in table. Updating columns...

--- Batch Process (5/612) ---

Processing Artist Request: 'A.R. Rahman'
Quer

### Chart Masters Artists Fuzzy Merge

In [16]:
def get_best_artist_match(artist_name, choices, threshold=90):
    """Uses strict full-string comparison to avoid substring false positives."""
    if pd.isna(artist_name): 
        return None
        
    # fuzz.Ratio forces a strict, character-by-character full string comparison
    match = process.extractOne(artist_name, choices, scorer=fuzz.ratio)
    
    if match and match[1] >= threshold:
        return match[0] # Returns the matched string
    return artist_name # Return original if no valid match

In [17]:
print("\n=== MERGING CHARTMASTERS ARTIST DATA (FUZZY) ===")

artists_file = CHART_MASTERS_ARTISTS
df_cm_artists = pd.read_csv(artists_file)

# 1. Prepare CHARTMASTERS DATA
df_cm_artists_clean = df_cm_artists[['Ranking', 'Artist', 'Total EAS']].copy()
df_cm_artists_clean = df_cm_artists_clean.rename(columns={
    'Ranking': 'cm_artist_ranking',
    'Total EAS': 'cm_artist_eas'
})

# 2. Normalization helper
def normalize_artist_names(series):
    """Vectorized cleaning to strip punctuation and extra spaces for artists."""
    return (
        series.astype(str).str.lower()
        .str.replace(r'[^\w\s]', '', regex=True) # Removes punctuation (e.g., P!nk -> pnk)
        .str.strip()
    )

# 3. Apply normalization
df_artists['match_artist'] = normalize_artist_names(df_artists['artist_name'])
df_cm_artists_clean['match_artist'] = normalize_artist_names(df_cm_artists_clean['Artist'])

# 4. Fuzzy matching
# Extract list of clean ChartMasters artists to match against
cm_artist_choices = df_cm_artists_clean['match_artist'].dropna().unique().tolist()

# Apply fuzzy mapping to main dataset
# Using a strict threshold (90) to prevent false positives
df_artists['match_artist_cm'] = df_artists['match_artist'].apply(
    lambda x: get_best_artist_match(x, cm_artist_choices, threshold=90)
)

# 5. Clean up
df_cm_artists_clean['cm_artist_ranking'] = df_cm_artists_clean['cm_artist_ranking'].astype(str).str.replace(',', '', regex=False)
df_cm_artists_clean['cm_artist_eas'] = df_cm_artists_clean['cm_artist_eas'].astype(str).str.replace(',', '', regex=False)

# Convert to nullable integers ('Int64')
df_cm_artists_clean['cm_artist_ranking'] = pd.to_numeric(df_cm_artists_clean['cm_artist_ranking'], errors='coerce').astype('Int64')
df_cm_artists_clean['cm_artist_eas'] = pd.to_numeric(df_cm_artists_clean['cm_artist_eas'], errors='coerce').astype('Int64')

# Drop any duplicate artists from ChartMasters
df_cm_artists_clean = df_cm_artists_clean.drop_duplicates(subset=['match_artist'])

# 6. Merge (using the targeted fuzzy match column)
cols_to_merge = ['match_artist', 'cm_artist_ranking', 'cm_artist_eas']
df_artists = pd.merge(
    df_artists, 
    df_cm_artists_clean[cols_to_merge], 
    left_on='match_artist_cm', 
    right_on='match_artist', 
    how='left'
)

# Drop the temporary matching columns
df_artists = df_artists.drop(columns=['match_artist_x', 'match_artist_y', 'match_artist'], errors='ignore')

print("Fuzzy Merge Complete! Here is a sample of the updated Artists table:")
print(df_artists[['artist_name', 'cm_artist_ranking', 'cm_artist_eas', 'match_artist_cm']].head(10))


=== MERGING CHARTMASTERS ARTIST DATA (FUZZY) ===
Fuzzy Merge Complete! Here is a sample of the updated Artists table:
         artist_name  cm_artist_ranking  cm_artist_eas    match_artist_cm
0             *NSYNC                236       48349000              nsync
1                -M-               <NA>           <NA>                  m
2               A-ha                280       39472000                aha
3       A. R. Rahman                335       31764000          ar rahman
4        A.R. Rahman                335       31764000          ar rahman
5        AUDREY NUNA               <NA>           <NA>        audrey nuna
6             AURORA               <NA>           <NA>             aurora
7            Aaliyah                440       19827000            aaliyah
8  Abraham Alexander               <NA>           <NA>  abraham alexander
9       Abraham Ivan               <NA>           <NA>       abraham ivan


In [18]:
print("ChartMasters missing:", df_artists['cm_artist_ranking'].isna().sum())

ChartMasters missing: 506


### Export Data Files

In [19]:
# Collect Dataframes
df_extracted_facts = df_songs_films.copy()
df_extracted_songs = df_songs_ranked.copy()
df_extracted_movies = df_movies_enriched.copy()
df_extracted_artists = df_artists.copy()

# Export Dataframes
df_extracted_facts.to_pickle('data/raw_facts_aux.pkl')
df_extracted_songs.to_pickle('data/raw_songs_aux.pkl')
df_extracted_movies.to_pickle('data/raw_movies_aux.pkl')
df_extracted_artists.to_pickle('data/raw_artists_aux.pkl')

print("Dataframes successfully saved!")

Dataframes successfully saved!
